In [1]:
import sys
sys.path.append('../')

import importlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import linkage

from src.preprocessing import (
    load_cluster_data, clean_customer_data,
    engineer_customer_features, encode_categorical, scale_features
)
import src.utils as _utils
importlib.reload(_utils)
from src.utils import COLORS, plot_silhouette, show_figure
import src.display as _display
importlib.reload(_display)
from src.display import (
    init_notebook_theme, show_hero, show_section, show_insight,
    show_info, show_warning, show_success, show_metrics_row,
    show_findings_list, show_badge, show_table_html, show_architecture_card,
)
init_notebook_theme()

DATA_PATH = '../data/raw/data_cluster.csv'
RANDOM_STATE = 42


In [2]:
show_hero(
    title="Exercice 2 — Segmentation intelligente des clients",
    objective="Identifier des profils clients pour optimiser les campagnes marketing.",
    plan_items=[
        "EDA", "Nettoyage & features", "Prétraitement clustering",
        "Clustering multi-algos", "Évaluation", "Interprétation métier", "Recommandations",
    ],
)
show_section("1. Analyse exploratoire (EDA)")

In [3]:
df = load_cluster_data(DATA_PATH)
show_metrics_row({"Lignes": f"{df.shape[0]:,}", "Colonnes": len(df.columns)})
print(df.columns.tolist())

['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']


In [4]:
print("--- Informations ---")
df.info()
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n--- Statistiques ---")
df.describe()

--- Informations ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 29 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   2240 non-null   int64  
 1   Year_Birth           2240 non-null   int64  
 2   Education            2240 non-null   object 
 3   Marital_Status       2240 non-null   object 
 4   Income               2216 non-null   float64
 5   Kidhome              2240 non-null   int64  
 6   Teenhome             2240 non-null   int64  
 7   Dt_Customer          2240 non-null   object 
 8   Recency              2240 non-null   int64  
 9   MntWines             2240 non-null   int64  
 10  MntFruits            2240 non-null   int64  
 11  MntMeatProducts      2240 non-null   int64  
 12  MntFishProducts      2240 non-null   int64  
 13  MntSweetProducts     2240 non-null   int64  
 14  MntGoldProds         2240 non-null   int64  
 15  NumDealsPurchases

,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
count,2240.000000,2240.000000,2216.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,...,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.000000,2240.0,2240.0,2240.000000
mean,5592.159821,1968.805804,52247.251354,0.444196,0.506250,49.109375,303.935714,26.302232,166.950000,37.525446,...,5.316518,0.072768,0.074554,0.072768,0.064286,0.013393,0.009375,3.0,11.0,0.149107
std,3246.662198,11.984069,25173.076661,0.538398,0.544538,28.962453,336.597393,39.773434,225.715373,54.628979,...,2.426645,0.259813,0.262728,0.259813,0.245316,0.114976,0.096391,0.0,0.0,0.356274
min,0.000000,1893.000000,1730.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
25%,2828.250000,1959.000000,35303.000000,0.000000,0.000000,24.000000,23.750000,1.000000,16.000000,3.000000,...,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
50%,5458.500000,1970.000000,51381.500000,0.000000,0.000000,49.000000,173.500000,8.000000,67.000000,12.000000,...,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
75%,8427.750000,1977.000000,68522.000000,1.000000,1.000000,74.000000,504.250000,33.000000,232.000000,50.000000,...,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,3.0,11.0,0.000000
max,11191.000000,1996.000000,666666.000000,2.000000,2.000000,99.000000,1493.000000,199.000000,1725.000000,259.000000,...,20.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,3.0,11.0,1.000000


In [5]:
# Distribution des variables clés
key_vars = ['Income', 'Age' if 'Age' in df.columns else 'Year_Birth',
            'MntWines', 'MntMeatProducts', 'NumWebPurchases', 'NumStorePurchases']
key_vars = [v for v in key_vars if v in df.columns][:6]

fig = make_subplots(rows=2, cols=3, subplot_titles=key_vars)
for i, var in enumerate(key_vars):
    row, col = i // 3 + 1, i % 3 + 1
    fig.add_trace(go.Histogram(x=df[var], marker_color=COLORS['primary'], name=var, showlegend=False), row=row, col=col)

fig.update_layout(title_text='Distribution des variables principales', height=650, width=1100)
show_figure(fig, '../reports/figures/ex2_distributions.png', width=1100, height=650)


In [6]:
# Variables catégorielles
cat_vars = [v for v in ['Education', 'Marital_Status'] if v in df.columns]

fig = make_subplots(rows=1, cols=max(len(cat_vars), 1), subplot_titles=cat_vars)
for i, var in enumerate(cat_vars):
    counts = df[var].value_counts()
    fig.add_trace(
        go.Bar(x=counts.index.astype(str), y=counts.values, marker_color=COLORS['primary'], showlegend=False),
        row=1, col=i + 1,
    )

fig.update_layout(height=450, width=1000)
show_figure(fig, '../reports/figures/ex2_categorical.png', width=1000, height=450)


In [7]:
# Dépenses par canal d'achat
spend_cols = [c for c in df.columns if c.startswith('Mnt')]
channel_cols = [c for c in df.columns if 'Purchases' in c]

spend_mean = df[spend_cols].mean().sort_values()
channel_mean = df[channel_cols].mean().sort_values()

fig = make_subplots(rows=1, cols=2, subplot_titles=['Dépenses moyennes par catégorie', 'Achats moyens par canal'])
fig.add_trace(go.Bar(x=spend_mean.values, y=spend_mean.index, orientation='h', marker_color=COLORS['primary']), row=1, col=1)
fig.add_trace(go.Bar(x=channel_mean.values, y=channel_mean.index, orientation='h', marker_color=COLORS['accent']), row=1, col=2)
fig.update_layout(height=450, width=1000)
show_figure(fig, '../reports/figures/ex2_spending_channels.png', width=1000, height=450)


In [8]:
show_section("2. Nettoyage & Feature Engineering")

In [9]:
df_clean = clean_customer_data(df)
df_clean = engineer_customer_features(df_clean)

print(f"Lignes avant nettoyage : {len(df)}")
print(f"Lignes après nettoyage : {len(df_clean)}")
print(f"\nNouvelles features créées : Age, Children, TotalSpend, TotalPurchases")

Lignes avant nettoyage : 2240
Lignes après nettoyage : 2215

Nouvelles features créées : Age, Children, TotalSpend, TotalPurchases


In [10]:
# Matrice de corrélation
num_cols = df_clean.select_dtypes(include=np.number).columns
corr_matrix = df_clean[num_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
corr_masked = corr_matrix.mask(mask)

fig = go.Figure(go.Heatmap(
    z=corr_masked.values, x=corr_masked.columns, y=corr_masked.columns,
    colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
))
fig.update_layout(title='Matrice de corrélation', height=800, width=1000)
show_figure(fig, '../reports/figures/ex2_correlation.png', width=1000, height=800)


In [11]:
show_section("3. Prétraitement pour clustering")

In [12]:
# Sélection des features pour le clustering
cat_cols = ['Education', 'Marital_Status']
drop_cols = ['ID', 'Dt_Customer', 'Year_Birth'] + cat_cols

df_encoded = encode_categorical(df_clean, cat_cols)
feature_cols = [c for c in df_encoded.select_dtypes(include=np.number).columns
                if c not in drop_cols]

X = df_encoded[feature_cols].copy()
print(f"Features pour clustering ({len(feature_cols)}) :\n{feature_cols}")

Features pour clustering (28) :
['Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response', 'Age', 'Children', 'TotalSpend', 'TotalPurchases']


In [13]:
X_scaled, scaler = scale_features(X)

# PCA pour visualisation 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f"Variance expliquée par les 2 premières composantes : {pca.explained_variance_ratio_.sum():.1%}")

# PCA pour réduction dimensionnelle (95% de variance)
pca_full = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)
print(f"Composantes pour 95% variance : {X_pca_full.shape[1]}")

Variance expliquée par les 2 premières composantes : 42.9%
Composantes pour 95% variance : 19


In [14]:
# Scree plot
pca_all = PCA(random_state=RANDOM_STATE)
pca_all.fit(X_scaled)
components = list(range(1, len(pca_all.explained_variance_ratio_) + 1))
cumvar = np.cumsum(pca_all.explained_variance_ratio_)

fig = make_subplots(rows=1, cols=2, subplot_titles=['Scree Plot', 'Variance cumulative'])
fig.add_trace(go.Scatter(x=components, y=pca_all.explained_variance_ratio_, mode='lines+markers', name='Variance'), row=1, col=1)
fig.add_trace(go.Scatter(x=components, y=cumvar, mode='lines+markers', name='Cumul', line_color=COLORS['accent']), row=1, col=2)
fig.add_hline(y=0.95, line_dash='dash', line_color='gray', annotation_text='95%', row=1, col=2)
fig.update_xaxes(title_text='Composante')
fig.update_yaxes(title_text='Variance expliquée', row=1, col=1)
fig.update_layout(height=450, width=1000)
show_figure(fig, '../reports/figures/ex2_pca_scree.png', width=1000, height=450)


In [15]:
show_section("4. Clustering")

In [16]:
k_range = range(2, 11)
inertias, silhouette_scores, db_scores = [], [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca_full)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_pca_full, labels))
    db_scores.append(davies_bouldin_score(X_pca_full, labels))
best_db_k = list(k_range)[np.argmin(db_scores)]
fig = make_subplots(rows=1, cols=3, subplot_titles=['Elbow', 'Silhouette', 'Davies-Bouldin'])
fig.add_trace(go.Scatter(x=list(k_range), y=inertias, mode='lines+markers', line_color=COLORS['primary']), row=1, col=1)
plot_silhouette(silhouette_scores, list(k_range), fig=fig, row=1, col=2)
fig.add_trace(go.Scatter(x=list(k_range), y=db_scores, mode='lines+markers', line_color=COLORS['accent']), row=1, col=3)
fig.update_layout(height=450, width=1200)
show_figure(fig, '../reports/figures/ex2_kmeans_selection.png', width=1200, height=450)
BEST_K = list(k_range)[np.argmax(silhouette_scores)]
show_metrics_row({"k optimal": BEST_K, "Silhouette max": f"{max(silhouette_scores):.4f}", "DB min k": best_db_k})

In [17]:
# K-Means final
kmeans = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca_full)

# Clustering hiérarchique
agglo = AgglomerativeClustering(n_clusters=BEST_K, linkage='ward')
agglo_labels = agglo.fit_predict(X_pca_full)

# GMM
gmm = GaussianMixture(n_components=BEST_K, random_state=RANDOM_STATE, covariance_type='full')
gmm_labels = gmm.fit_predict(X_pca_full)

# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=10)
dbscan_labels = dbscan.fit_predict(X_pca_full)
n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
print(f"DBSCAN : {n_dbscan_clusters} clusters, {(dbscan_labels == -1).sum()} outliers")

DBSCAN : 0 clusters, 2215 outliers


In [18]:
# Comparaison visuelle des algorithmes
algo_results = [
    (kmeans_labels, 'K-Means'),
    (agglo_labels, 'Agglomerative'),
    (gmm_labels, 'GMM'),
    (dbscan_labels, 'DBSCAN'),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[t for _, t in algo_results])
positions = [(1, 1), (1, 2), (2, 1), (2, 2)]

for (labels, title), (row, col) in zip(algo_results, positions):
    fig.add_trace(
        go.Scatter(
            x=X_pca[:, 0], y=X_pca[:, 1], mode='markers',
            marker=dict(color=labels, colorscale='Tab10', size=5, opacity=0.6, showscale=(row == 2 and col == 2)),
            name=title, showlegend=False,
        ),
        row=row, col=col,
    )
    fig.update_xaxes(title_text='PC1', row=row, col=col)
    fig.update_yaxes(title_text='PC2', row=row, col=col)

fig.update_layout(title='Comparaison des algorithmes de clustering', height=800, width=1000)
show_figure(fig, '../reports/figures/ex2_clustering_comparison.png', width=1000, height=800)


ValueError: 
    Invalid value of type 'builtins.str' received for the 'colorscale' property of scatter.marker
        Received value: 'Tab10'

    The 'colorscale' property is a colorscale and may be
    specified as:
      - A list of colors that will be spaced evenly to create the colorscale.
        Many predefined colorscale lists are included in the sequential, diverging,
        and cyclical modules in the plotly.colors package.
      - A list of 2-element lists where the first element is the
        normalized color level value (starting at 0 and ending at 1),
        and the second item is a valid color string.
        (e.g. [[0, 'green'], [0.5, 'red'], [1.0, 'rgb(0, 0, 255)']])
      - One of the following named colorscales:
            ['aggrnyl', 'agsunset', 'algae', 'amp', 'armyrose', 'balance',
             'blackbody', 'bluered', 'blues', 'blugrn', 'bluyl', 'brbg',
             'brwnyl', 'bugn', 'bupu', 'burg', 'burgyl', 'cividis', 'curl',
             'darkmint', 'deep'

In [ ]:
# Dendrogramme (clustering hiérarchique)
from plotly.figure_factory import create_dendrogram

sample_idx = np.random.choice(len(X_pca_full), 300, replace=False)
fig = create_dendrogram(
    X_pca_full[sample_idx], linkagefun=lambda x: linkage(x, method='ward'),
    orientation='bottom', labels=None,
)
fig.update_layout(title='Dendrogramme (échantillon 300 clients)', xaxis_title='Clients', yaxis_title='Distance Ward', height=500, width=1200)
show_figure(fig, '../reports/figures/ex2_dendrogram.png', width=1200, height=500)


In [ ]:
show_section("5. Évaluation & comparaison")

In [ ]:
# Tableau de comparaison
eval_results = []
valid_algos = [
    ('K-Means', kmeans_labels),
    ('Agglomerative', agglo_labels),
    ('GMM', gmm_labels),
]
if n_dbscan_clusters > 1:
    mask = dbscan_labels != -1
    valid_algos.append(('DBSCAN', dbscan_labels))

for name, labels in valid_algos:
    try:
        sil = silhouette_score(X_pca_full, labels)
        db = davies_bouldin_score(X_pca_full, labels)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        eval_results.append({'Algorithme': name, 'N clusters': n_clust,
                              'Silhouette ↑': round(sil, 4), 'Davies-Bouldin ↓': round(db, 4)})
    except Exception as e:
        print(f"{name} : {e}")

eval_df = pd.DataFrame(eval_results).set_index('Algorithme')
print(eval_df.to_string())

In [ ]:
show_section("6. Interprétation métier des profils")

In [ ]:
# Ajouter les labels K-Means au dataframe original
df_result = df_clean.copy()
df_result['Cluster'] = kmeans_labels

# Profil moyen par cluster
profile_cols = ['Income', 'Age', 'TotalSpend', 'TotalPurchases',
                'MntWines', 'MntMeatProducts', 'NumWebPurchases',
                'NumStorePurchases', 'Recency', 'Children']
profile_cols = [c for c in profile_cols if c in df_result.columns]

cluster_profiles = df_result.groupby('Cluster')[profile_cols].mean().round(1)
print("=== Profils moyens par cluster ===")
print(cluster_profiles.T.to_string())

In [ ]:
# Heatmap des profils normalisés
from sklearn.preprocessing import MinMaxScaler

profile_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(cluster_profiles),
    index=cluster_profiles.index,
    columns=cluster_profiles.columns,
)

fig = go.Figure(go.Heatmap(
    z=profile_norm.T.values,
    x=[str(c) for c in profile_norm.index],
    y=profile_norm.columns,
    colorscale='YlOrRd',
    text=cluster_profiles.T.round(0).astype(int).values,
    texttemplate='%{text}',
))
fig.update_layout(title='Profils clients par cluster (normalisés)', xaxis_title='Cluster', height=500, width=1100)
show_figure(fig, '../reports/figures/ex2_cluster_profiles.png', width=1100, height=500)


In [ ]:
# Nommer les clusters (à adapter selon les résultats)
# Exemple de nommage basé sur les profils observés :
cluster_names = {
    0: 'Clients Premium',       # Hauts revenus, fortes dépenses
    1: 'Clients Digitaux',      # Achats web élevés, jeunes
    2: 'Promo-sensibles',       # Nombreux achats promotionnels
    3: 'Clients Dormants',      # Faibles dépenses, peu actifs
}
# Adapter cluster_names selon BEST_K et les profils réels
if BEST_K <= len(cluster_names):
    df_result['Profil'] = df_result['Cluster'].map(
        {k: v for k, v in cluster_names.items() if k < BEST_K}
    )

print("Taille des clusters :")
print(df_result['Cluster'].value_counts().sort_index())

In [ ]:
# Radar chart des profils
radar_cols = [c for c in ['Income', 'TotalSpend', 'NumWebPurchases', 'NumStorePurchases',
                          'NumDealsPurchases', 'Recency'] if c in cluster_profiles.columns]

from sklearn.preprocessing import MinMaxScaler

norm_profiles = MinMaxScaler().fit_transform(cluster_profiles[radar_cols])
palette = px.colors.qualitative.Tab10

fig = go.Figure()
for i, row in enumerate(norm_profiles[:min(BEST_K, 4)]):
    values = row.tolist() + [row[0]]
    theta = radar_cols + [radar_cols[0]]
    fig.add_trace(go.Scatterpolar(
        r=values, theta=theta, fill='toself', name=cluster_names.get(i, f'Cluster {i}'),
        line_color=palette[i % len(palette)],
    ))

fig.update_layout(title='Radar — Profils clients', polar=dict(radialaxis=dict(visible=True, range=[0, 1])), height=500, width=1000)
show_figure(fig, '../reports/figures/ex2_radar_profiles.png', width=1000, height=500)


In [ ]:
show_section("7. Recommandations business")

In [ ]:
# Taux de réponse aux campagnes par cluster
cmp_cols = [c for c in df_result.columns if c.startswith('AcceptedCmp') or c == 'Response']

if cmp_cols:
    df_result['CampaignResponse'] = df_result[cmp_cols].sum(axis=1)
    campaign_by_cluster = df_result.groupby('Cluster')['CampaignResponse'].mean()

    fig = go.Figure(go.Bar(
        x=campaign_by_cluster.index.astype(str), y=campaign_by_cluster.values,
        marker_color=COLORS['primary'],
    ))
    fig.update_layout(
        title='Réponse moyenne aux campagnes marketing par cluster',
        xaxis_title='Cluster', yaxis_title='Taux de réponse moyen', height=450, width=900,
    )
    show_figure(fig, '../reports/figures/ex2_campaign_response.png', width=900, height=450)


In [ ]:
show_section("Synthèse Exercice 2")
profiles_df = pd.DataFrame([
    {"Cluster": 0, "Profil": "Premium", "Actions": "Fidélité, offres exclusives"},
    {"Cluster": 1, "Profil": "Digitaux", "Actions": "Email, app mobile"},
    {"Cluster": 2, "Profil": "Promo-sensibles", "Actions": "Coupons ciblés"},
])
show_table_html(profiles_df.set_index("Cluster"), title="Profils (à adapter)")
show_findings_list([
    f"k optimal = {BEST_K} (Silhouette)",
    "Personnaliser les campagnes par profil cluster",
    "Recalcul mensuel avec suivi Silhouette",
], title="Recommandations stratégiques")